# Droplet rebound simulation (gauthier experiments) - Run

This notebook is used to run the actual simulations for the droplet rebound test case. The simulation will be restarted from a previously computed initial state, see `DropletReboundInit.ipynb`

In [1]:
//#r "BoSSSpad/BoSSSpad.dll"
#r "..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
using System.IO;
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [10]:
public static double RoundSignificantDigit(double input, int digit = 1, bool floor = false) {

    if (digit < 1)
        throw new ArgumentException("digit must be greater than zero");

    if (input == 0.0)
        return input;

    int precision = 0;
    double val = input - Math.Round(input, 0);
    while (Math.Abs(val) < (double)(10.Pow(digit - 1))) {
        val *= 10.0;
        precision++;
    }

    if (floor) {
        var power = Math.Pow(10, precision);
        return Math.Floor(input * power) / power;
    } else {
        return Math.Round(input, precision);
    }    
}

In [13]:
RoundSignificantDigit(0.00001963, 3, true)

1.96E-05

In [40]:
Console.WriteLine("Execution Date/time is " + DateTime.Now);

Execution Date/time is 9/3/2025 10:20:48 AM


In [41]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [42]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [43]:
var myBatch = ExecutionQueues[1];

if(userName.Equals(@"FDY\JenkinsCI"))
    myBatch = GetDefaultQueue();

myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [44]:
BoSSSshell.WorkflowMgm.Init("DropletRebound_Gauthier", myBatch);

Project name is set to 'DropletRebound_Gauthier'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier'.


In [45]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setting up restart

Choose if this notebook should do a restart. 

In [46]:
bool restartRun = true;

In [47]:
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\DropletRebound_Gauthier");
//OpenOrCreateDatabase(@"D:\local\DropletRebound_Gauthier");

In [48]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2_DongBC_restart3_Agglom02*	8/27/2025 12:52:42 PM	92536e2a...
#1: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_Mumps_restart2*	8/26/2025 9:40:37 AM	31b3fb2d...
#2: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_ReInit4_Mumps_restart1*	8/26/2025 9:29:43 AM	a7bc5cb5...
#3: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_ReInit4_wallBC_Mumps*	8/25/2025 3:21:44 PM	ef27dbc0...
#4: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Algoim*	8/25/2025 12:39:30 PM	e9e4304e...
#5: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_QuadOrder2*	8/25/2025 11:53:50 AM	7e7a00a3...
#6: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Agglom02*	8/25/2025 11:51:56 AM	ca919415...
#7: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycl

In [49]:
var sess = wmg.Sessions.Pick(6);
sess

DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Agglom02*	8/25/2025 11:51:56 AM	ca919415...

In [50]:
//sess.Timesteps.Last().GetField("PhiDG").Basis.Degree
//sess.Timesteps.First().TimeStepNumber.MajorNumber

In [ ]:
//sess.Export().WithSupersampling(2).Do();
//sess.GetSessionDirectory()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound_Gauthier__DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Agglom02__ca919415-fcb4-434f-b83a-e6263c4de16a


In [52]:
//var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2_DongBC")); 
//var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR0_k3_ReI4"));
//var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_10x12x8"));
//var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_10x12x8_AMR0_k3_expKcycleMumps")); 
var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps"));
selection

#0: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart1*	7/28/2025 12:11:26 PM	9e396d10...
#1: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newBndFlx_restart1*	7/24/2025 12:23:23 PM	2f32acf4...


In [53]:
//selection.Pick(0).RestartedFrom

In [54]:
//selection.Pick(2).Timesteps.Last().Export().WithSupersampling(2).Do();

In [55]:
//BoSSSshell.WorkflowMgm.Sessions
//sessions.Pick(32).RestartedFrom

In [56]:
//BoSSSshell.WorkflowMgm.Sessions.Pick(0).Delete(true)
//selection.Pick(0).Copy(databases.Pick(4))

In [57]:
// // check for backup on userspace database
// var backupDB = OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");
// List<ISessionInfo> noBackUpSessions = new List<ISessionInfo>();
// foreach(var sess in sessions) {
//     if(!backupDB.Sessions.Contains(sess)) {
//         noBackUpSessions.Add(sess);
//     }
// }
// noBackUpSessions

In [58]:
//noBackUpSessions.MoveAll(backupDB);

In [59]:
ISessionInfo restartSession;
if(restartRun == true) {
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR1_k3_ReI4_restart4")).Single();
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x12x8AMR1_k2_ReI4_restart2")).Single();
    //restartSession = selection.Last();
    //restartSession = selection.First();
    restartSession = selection.Pick(0);
    //restartSession = databases.Pick(0).Sessions.Pick(0);
}
restartSession

DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart1*	7/28/2025 12:11:26 PM	9e396d10...

In [60]:
List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
restartSessionList.Add(restartSession);
Guid restartID = restartSession.RestartedFrom;
while(restartID != Guid.Empty) {
    var rSess = sessions.Where(sess => sess.ID == restartID).Single();
    restartSessionList.Add(rSess);
    restartID = rSess.RestartedFrom;
}
restartSessionList

#0: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart1*	7/28/2025 12:11:26 PM	9e396d10...
#1: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_ReInit4_expKcycleMumps_newBndFlx_noFailOnSolverFail*	7/16/2025 4:04:26 PM	ede942a0...


In [61]:
//restartSessionList.Pick(1).GetSessionDirectory()

In [62]:
//restartSessionList.CopyAll(databases.Pick(1));
//restartSessionList.Pick(0).Timesteps.Last().Export().WithSupersampling(2).Do();

In [63]:
//restartSession.GetSessionDirectory()
//restartSession.Copy(databases.Pick(1));

In [64]:
//restartSession.Timesteps.Count

In [65]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(2).Do()

automated naming of restart sessions

In [66]:
string restartName = (restartSession != null) ? String.Empty : "noRestart";

In [67]:
if(restartSession != null) {
Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
procSIs.Push(restartSession);
var currSI = restartSession;
var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
while(!rSIs.IsNullOrEmpty()) {
    var rSI = rSIs.Single();
    procSIs.Push(rSI);
    currSI = rSI;
    rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
}
int restartNum = procSIs.Count;

string orgName = restartSession.Name;
if (restartNum == 1) {
    //restartName = orgName.Substring(0, orgName.Length - 10); // remoinvg _InitState
//} else if (restartNum == 1){
    restartName = orgName.Substring(0, orgName.Length) + "_restart1"; // + restartNum; 
} else {
    restartName = orgName.Substring(0, orgName.Length - 1) + restartNum; 
}
}
restartName

DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart2

In [68]:
restartName = "DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_Mumps_restart2";

## Boundary Conditions

0: dong outflow, 1: velocity inlet

In [69]:
static int BC_tag2_top = 0;
static int BC_tag4_front = 0;

## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [70]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial_back, double l_radial_front, double l_upstream, double l_downstream, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - l_radial_back, radiusOP + l_radial_front, res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_upstream, l_downstream, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "wall_rotatingDisk");

    if (BC_tag2_top == 0)
        grd.EdgeTagNames.Add(2, "Dong_OutFlow_top");
    if (BC_tag2_top == 1)
        grd.EdgeTagNames.Add(2, "velocity_inlet_top");

    grd.EdgeTagNames.Add(3, "velocity_inlet_back");

    if (BC_tag4_front == 0)
        grd.EdgeTagNames.Add(4, "Dong_OutFlow_front");
    if (BC_tag4_front == 1)
        grd.EdgeTagNames.Add(4, "velocity_inlet_front");
   
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(6, "Dong_OutFlow_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + l_radial_back).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - l_radial_front).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_upstream)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_downstream)).Abs() <= 1e-8)
            et = 6;
        // if ((X.y + (l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 5;
        // if ((X.y - (2.0 * l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 6;

        return et;
    });    

    return grd;
}

In [71]:
static public GridCommons RotatingDiskSector_CartesianCutOutExtended(double radiusOP, double l_radial, double l_upstream, double l_downstream, double h_axial, int res_radial, int res_azimuthal, int res_axial, 
                                                                    int radialExtLayer = 0, bool bothRadialDir = false, int azimuthalExtLayer = 0, bool bothAzimuthalDir = false, int axialExtLayer = 0) {

            int maxLayer = (new int[] { radialExtLayer, azimuthalExtLayer, axialExtLayer }).Max();
            GridCommons.GridBox[] grdBoxes = new GridCommons.GridBox[maxLayer + 1];

            grdBoxes[maxLayer] = new GridCommons.GridBox( new double[] { radiusOP - (l_radial / 2.0), -l_upstream, 0.0 },
                new double[] { radiusOP + (l_radial / 2.0), l_downstream , h_axial},
                res_radial, res_azimuthal, res_axial );

            double dx_radial = l_radial / res_radial;
            double dx_azimuthal = (l_upstream + l_downstream) / res_azimuthal;
            double dx_axial = h_axial / res_axial;

            for (int l = 1; l <= maxLayer; l++) {
                var embBB = grdBoxes[(maxLayer + 1) - l].boundingBox;
                double xMin = embBB.Min[0];
                double xMax = embBB.Max[0];
                double yMin = embBB.Min[1];
                double yMax = embBB.Max[1];
                double zMin = 0.0;
                double zMax = embBB.Max[2];

                var cellsBB = grdBoxes[(maxLayer + 1) - l].numOfCells;

                int resExtLayer_radial = (cellsBB[0] / 2);
                if (l <= radialExtLayer) {
                    if (bothRadialDir) {
                        resExtLayer_radial += 2;
                        xMax += dx_radial * 2.0.Pow(l);
                        xMin -= dx_radial * 2.0.Pow(l);
                        // ensure 2:1
                        int rEL_radMod = (resExtLayer_radial / 2) % 2;
                        if (rEL_radMod == 1) {
                            resExtLayer_radial += 2;
                            xMax += dx_radial * 2.0.Pow(l);
                            xMin -= dx_radial * 2.0.Pow(l);
                        }
                    } else {
                        resExtLayer_radial += 1;
                        xMax += dx_radial * 2.0.Pow(l);
                        // ensure 2:1
                        int rEL_radMod = resExtLayer_radial % 2;
                        if (rEL_radMod == 1) {
                            resExtLayer_radial += 1;
                            xMax += dx_radial * 2.0.Pow(l);
                        }
                    }
                }

                int resExtLayer_azimuthal = (cellsBB[1] / 2);
                if (l <= azimuthalExtLayer) {
                    if (bothAzimuthalDir) {
                        resExtLayer_azimuthal += 2;
                        yMax += dx_azimuthal * 2.0.Pow(l);
                        yMin -= dx_azimuthal * 2.0.Pow(l);
                        // ensure 2:1
                        int rEL_aziMod = (resExtLayer_azimuthal / 2) % 2;
                        if (rEL_aziMod == 1) {
                            resExtLayer_azimuthal += 2;
                            yMax += dx_azimuthal * 2.0.Pow(l);
                            yMin -= dx_azimuthal * 2.0.Pow(l);
                        }
                    }
                    else {
                        resExtLayer_azimuthal += 1;
                        yMax += dx_azimuthal * 2.0.Pow(l);
                        // ensure 2:1
                        int rEL_aziMod = resExtLayer_azimuthal % 2;
                        if (rEL_aziMod == 1) {
                            resExtLayer_azimuthal += 1;
                            yMax += dx_azimuthal * 2.0.Pow(l);
                        }
                    }
                }

                int resExtLayer_axial = (cellsBB[2] / 2);
                if (l <= axialExtLayer) {
                    resExtLayer_axial += 1;
                    zMax += dx_axial * 2.0.Pow(l); 
                    // ensure 2:1
                    int rEL_axiMod = resExtLayer_axial % 2;
                    if (rEL_axiMod == 1) {
                        resExtLayer_axial += 1;
                        zMax += dx_axial * 2.0.Pow(l);  
                    }
                }

                grdBoxes[maxLayer - l] = new GridCommons.GridBox( new double[] { xMin, yMin, zMin },
                    new double[] { xMax, yMax, zMax },
                    resExtLayer_radial, resExtLayer_azimuthal, resExtLayer_axial);
            }


            var grd = Grid3D.HangingNodes3D(false, false, false, grdBoxes);
            grd.Name = $"RotatingDiskSector3D_CartesianCutOutExtended{radialExtLayer}{azimuthalExtLayer}{axialExtLayer}_{res_radial}x{res_azimuthal}x{res_axial}";

            grd.EdgeTagNames.Add(1, "wall_rotatingDisk");
            grd.EdgeTagNames.Add(2, "dong_outflow_top");
            grd.EdgeTagNames.Add(3, "velocity_inlet_back");
            grd.EdgeTagNames.Add(4, "Dong_OutFlow_front");
            grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
            grd.EdgeTagNames.Add(6, "dong_outflow_downstream");

            grd.DefineEdgeTags(delegate (Vector X) {
                byte et = 0;
                if ((X.z - grdBoxes[0].boundingBox.Min[2]).Abs() <= 1e-8)
                    et = 1;
                if ((X.z - grdBoxes[0].boundingBox.Max[2]).Abs() <= 1e-8)
                    et = 2;
                if ((X.x - grdBoxes[0].boundingBox.Min[0]).Abs() <= 1e-8)
                    et = 3;
                if ((X.x - grdBoxes[0].boundingBox.Max[0]).Abs() <= 1e-8)
                    et = 4;
                if ((X.y - grdBoxes[0].boundingBox.Min[1]).Abs() <= 1e-8)
                    et = 5;
                if ((X.y - grdBoxes[0].boundingBox.Max[1]).Abs() <= 1e-8)
                    et = 6;

                return et;
            });

            return grd;
}

## Simulation setup

In [72]:
double radiusOP = 90e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.2;  
double kinViscosity = 2.0e-5 / density; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double velOP = 38.0; 
double omega = velOP / radiusOP; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

0.00019867985355975657

Reynolds number at the point of injection

In [73]:
double reynolds = radiusOP / Lstar;
reynolds

452.990066116245

Boundary layer (BL) thickness

In [74]:
double zBL = 5.5;
double zBLstar = zBL * Lstar;
zBLstar

0.001092739194578661

In [75]:
double radiusDrop = 0.91e-3;
double topDropPos = zBLstar + (2.0*radiusDrop);
topDropPos

0.002912739194578661

Height of the computational domain 

In [76]:
double zTop = 20;
double zTopStar = zTop * Lstar;
zTopStar

0.003973597071195131

## Prescribed boundary conditions for the flow field 

The B.C. are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [77]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [78]:
MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffU.txt");
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");
MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffW.txt");
MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffP.txt");

In [79]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX();
vonKarmanHAM_velX.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity);
var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityY();
vonKarmanHAM_velY.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity);
var vonKarmanHAM_velZ = new RotatingDiskBoundaryLayerConditionsHAM_VelocityZ();
vonKarmanHAM_velZ.SetData(HAMcoeff_velW, omega, kinViscosity);
var vonKarmanHAM_P = new RotatingDiskBoundaryLayerConditionsHAM_PressureP();
vonKarmanHAM_P.SetData(HAMcoeff_P, omega, kinViscosity, density);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [80]:
omega

422.22222222222223

In [81]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velX = - rOnRotDisk * Math.Sin(theta) * omega;" +
    "return velX; } "
);

In [82]:
Formula RotatingDiskVelocityY = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velY = rOnRotDisk * Math.Cos(theta) * omega;" +
    "return velY; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [83]:
double velW_top = -0.88447342;
double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
velWstar_top

-0.07419586537108543

In [84]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.07419586537108543; } "
);

## Initial conditions for the injected droplet 

If not restarted from precomputed initial state with impact velocity for the droplet.

In [85]:
radiusOP

0.09

In [86]:
radiusDrop

0.00091

In [87]:
double initHeight = zBLstar + radiusDrop;
initHeight

0.0020027391945786613

In [88]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 0.09;" +
    "double radiusDrop = 0.91e-3;" +   
    "double initHeight = 1.5e-3;" + 
    "return Math.Sqrt((X[0] - radiusOP).Pow2() + X[1].Pow2() + (X[2] - initHeight).Pow2()) - radiusDrop; } "
);

In [89]:
Formula InitVelocity = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.23; } "
);

In [90]:
Formula GravityValue = new Formula(
    "GravZ",
    false,
    "double GravZ(double[] X) { return -9.81; } "
);

## Control settings

In [94]:
double densityDrop = 960;
double sigma = 21e-3;

int res_global = 8;
int AMRlevel_disk = 0;
int AMRlevel_drop = 1;
int AMRlevel_dropBL = 0;
int maxAMRlevel = (new int[] {AMRlevel_disk, AMRlevel_drop, AMRlevel_dropBL}).Max();
double hmin = (zTopStar / res_global) / (maxAMRlevel + 1);

int k = 3;

double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(densityDrop, density, sigma, hmin, k+1);
dt_sigma

2.987776171891897E-05

In [55]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [56]:
if (restartRun) {
    
    string sessionDir = restartSession.GetSessionDirectory();
    string path_obj = Path.Combine(sessionDir, "Control-obj.txt");

    string ctrlfileContent = File.ReadAllText(path_obj);

    var ctrl = (XNSE_Control)AppControl.Deserialize(ctrlfileContent).CloneAs();

    ctrl.InitialValues.Clear();
    ctrl.InitialValues_Evaluators.Clear();

    ctrl.GridGuid = Guid.Empty;
    int RestartTimestepNumber = 110; //restartSession.Timesteps.Last().TimeStepNumber.MajorNumber - 1;
    ctrl.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, RestartTimestepNumber);
    ctrl.ReInitTimestepIndex = RestartTimestepNumber;
    ctrl.NoOfTimesteps = 200;

    //ctrl.CutCellQuadratureType = CutCellQuadratureMethod.Algoim;

    //ctrl.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    ctrl.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    ctrl.StokesExtentionUseBCmap = false;

    //ctrl.AgglomerationThreshold = 0.2;

    ctrl.ReInitControl = new BoSSS.Solution.LevelSetTools.EllipticReInit.EllipticReInitAlgoControl() {
        UseAdaptiveReInit = false,
    };

    //ctrl.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();
    ctrl.LinearSolver = LinearSolverCode.direct_mumps.GetConfig();
    //ctrl.LinearSolver = LinearSolverCode.direct_pardiso.GetConfig();
    //ctrl.NonLinearSolver.Globalization = Newton.GlobalizationOption.Dogleg;

    ctrl.FailOnSolverFail = false;
    //ctrl.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();
    //ctrl.LinearSolver = LinearSolverCode.direct_mumps.GetConfig();

    // ctrl.ChangeBoundaryCondition("pressure_outlet_top", "Dong_OutFlow_top");
    // ctrl.ChangeBoundaryCondition("pressure_outlet_front", "Dong_OutFlow_front");
    // ctrl.ChangeBoundaryCondition("pressure_outlet_downstream", "Dong_OutFlow_downstream");

    ctrl.ChangeBoundaryCondition("velocity_inlet_rotatingDisk", "wall_rotatingDisk");
    //ctrl.BoundaryValues.Remove("velocity_inlet_rotatingDisk");
    ctrl.AddBoundaryValue("wall_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);
    ctrl.AddBoundaryValue("wall_rotatingDisk", "VelocityY#B", RotatingDiskVelocityY);

    {
        ctrl.AdaptiveMeshRefinement = true;
        //ctrl.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
        ctrl.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_dropBL });
    }

    //ctrl.DbPath = _DbPath;
    ctrl.savetodb = true;

    ctrl.SessionName = restartName;

    Controls.Add(ctrl);
}

In [57]:
if (!restartRun) {
var C = new XNSE_Control();

int DgLsDegree = k; int CgLsDegree = k+1;
C.SetFieldOptions(k, DgLsDegree, CgLsDegree); 
//C.SetDGdegree(k);
C.FieldOptions.Add(VariableNames.GravityZ, new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});


// physical parameters
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = 96e-3;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.IncludeConvection = true;

    
    // set grid
    double l_radial_back = (2.0 / 4.0) * zTopStar ; 
    double l_radial_front = (3.0 / 4.0) * zTopStar ; 
    double l_upstream = (1.0 / 2.0) * zTopStar;
    double l_downstream = (2.0 / 2.0) * zTopStar;
    double h_axial = zTopStar; 

    int res_radial = (5 * res_global) / 4; 
    //int extRes_radial = 2;
    int res_azimuthal = (3 * res_global) / 2;
    //int extRes_azimuthal = 3;
    int res_axial = 1 * res_global;
    //int extRes_axial = 2;

    //Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_upstream, l_downstream, h_axial, res_radial, res_azimuthal, res_axial);
    Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial_back, l_radial_front, l_upstream, l_downstream, h_axial, res_radial, res_azimuthal, res_axial);
    // GridCommons grd = RotatingDiskSector_CartesianCutOutExtended(radiusOP, l_radial, l_upstream, l_downstream, h_axial, 
    //                     res_radial, res_azimuthal, res_axial, extRes_radial, true, extRes_azimuthal, true, extRes_axial);
    C.SetGrid(grd);

    // initial conditions
    C.AddInitialValue("VelocityX#B", vonKarmanHAM_velX);
    C.AddInitialValue("VelocityY#B", vonKarmanHAM_velY);
    C.AddInitialValue("VelocityZ#B", vonKarmanHAM_velZ);
    // C.AddInitialValue("Pressure#B", vonKarman_P);

    C.AddInitialValue("Phi", PhiFunc);
    C.AddInitialValue("VelocityZ#A", InitVelocity);


C.AddInitialValue("GravityZ#A", GravityValue);
C.AddInitialValue("GravityZ#B", GravityValue);

// boundary conditions
C.AddBoundaryValue("wall_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);
C.AddBoundaryValue("wall_rotatingDisk", "VelocityY#B", RotatingDiskVelocityY);

if (BC_tag2_top == 1) {
    C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityZ#B", vonKarmanHAM_velZ);
}

if (BC_tag4_front == 1) {
    C.AddBoundaryValue("velocity_inlet_front", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityZ#B", vonKarmanHAM_velZ);
}

C.AddBoundaryValue("velocity_inlet_back", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_back", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_back", "VelocityZ#B", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityZ#B", vonKarmanHAM_velZ);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityY#B", vonKarman_velY);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityZ#B", vonKarman_velZ);
// C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#B", vonKarman_P);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

int ReInit = 4;
C.ReInitPeriod = ReInit;
// C.ReInitControl = new BoSSS.Solution.LevelSetTools.EllipticReInit.EllipticReInitAlgoControl() {
//     UseAdaptiveReInit = true,
// };
// C.PostprocessingModules.Add(new StokesExtensionEvolverLogging() { SolverStage = 2 });

//C.FailOnSolverFail = false;
// C.SkipSolveAndEvaluateResidual = true;

// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

//C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();
C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig();

//C.AgglomerationThreshold = 0.2;

// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.Timestepper_LevelSetHandling = LevelSetHandling.None;
// C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
//C.StokesExtentionUseBCmap = true;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 2.5e-5;
C.NoOfTimesteps = 400;
    
// {
//     C.AdaptiveMeshRefinement = true;
//     //C.activeAMRlevelIndicators.Add(new AMRwithGaussCheck() { maxRefinementLevel = AMRlevel_drop });
//     //C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
//     //C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_dropBL });
//     double minResAMR = (2.0 * h_axial / ((double)res_axial)) * 1.5;
//     C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundaryByResolution(new byte[] { 1 }, minResAMR)  { maxRefinementLevel = 2 });
//     C.AMR_startUpSweeps = 3;
// }

// C.PostprocessingModules.Add(new EnergyLogging());
C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers";
    
if (restartRun) 
    C.SessionName = restartName;
else {
    //C.SessionName = $"DropletReboundGauthier_{res_radial}x{res_azimuthal}x{res_axial}ExtGrid{extRes_radial}{extRes_azimuthal}{extRes_axial}BothDir_AMR{AMRlevel_drop}_k{k}_DongBC";
    C.SessionName = $"DropletReboundGauthier_{res_radial}x{res_azimuthal}x{res_axial}_AMR{AMRlevel_drop}_k{k}_ReInit4_wallBC_Mumps";
}
    
    
Controls.Add(C);
}

## Launch job

In [58]:
Controls.Select(C => C.SessionName)

[ DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_Mumps_restart2 ]

In [59]:
//Controls.Pick(0).FieldOptions.Get("PhiDG")

In [60]:
restartSession

DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart1*	7/28/2025 12:11:26 PM	9e396d10...

In [61]:
myBatch.Name

FDY-WindowsHPC

In [62]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 6;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    oneJob.Activate(myBatch); 
}

 ------------ MSHPC FailedOrCanceled; original Failed
Deployments so far (2): (Job token: 121977, FailedOrCanceled 'DropletRebound_Gauthier-XNSE_Solver2025Aug25_140042.711265' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FailedOrCanceled), (Job token: 121978, FailedOrCanceled 'DropletRebound_Gauthier-XNSE_Solver2025Aug25_141215.803339' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FailedOrCanceled);
Success: 0
job submit count: 2
Note: Job was deployed (2) number of times, all failed; RetryCount is 3, so try once more.
Hint: want to re-activate the job.
Job is FailedOrCanceled, but retry count is set to 3 and only 2 tries yet - trying once more...
Deploying job DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_Mumps_restart2 ... 


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletRebound_Gauthier-XNSE_Solver2025Aug26_094020.960662
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


In [63]:
wmg.Sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_Mumps_restart2*	8/25/2025 2:01:11 PM	99e76d24...
#1: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Algoim*	8/25/2025 12:39:30 PM	e9e4304e...
#2: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_QuadOrder2*	8/25/2025 11:53:50 AM	7e7a00a3...
#3: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Agglom02*	8/25/2025 11:51:56 AM	ca919415...
#4: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_Diag2*	8/25/2025 11:04:32 AM	038a9400...
#5: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_testStokes3_adaptiveReInit*	8/25/2025 9:45:00 AM	b1bfd4c5...
#6: DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR0_k3_expKcycle_noReInit_restart1_1thread_noCaching*	8/22/2025 2:59:58 PM	bf6d4479...
#7: DropletRebound_Gauthier	DropletReboundGauthi

In [65]:
var sess = wmg.Sessions.Pick(42);
sess

DropletRebound_Gauthier	DropletReboundGauthier_10x12x8_AMR1_k3_ReInit4_expKcycleMumps_newLSupdate_restart1*	7/28/2025 12:11:26 PM	9e396d10...

In [66]:
//sess.Timesteps.Last().GetField("Phi").Basis.Degree
sess.GetSessionDirectory()
//sess.Timesteps.Last()

\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier\sessions\9e396d10-bea1-4caf-b94c-8c08b1b34f39

In [68]:
sess.Export().WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound_Gauthier__DropletReboundGauthier_10x12x8_AMR0_k3_testStokes3_adaptiveReInit__b1bfd4c5-8a06-42d8-b0e6-a703dfe89efb


In [121]:
//sess.Delete(true);

Session ad004d64-ba07-4a22-b160-2757eae3d891 deleted.


## Wait for Completion and Check Job Status

In [ ]:
double hrsToWait = 0.25;

In [ ]:
wmg.BlockUntilAllJobsTerminate(hrsToWait*3600);

unable to determine job status - unknown
unable to determine job status - unknown
Timeout.


In [ ]:
var JobStat = Controls.Select(ctrl => ctrl.GetJob().Status).ToArray();
JobStat

index,value
0,InProgress


In [ ]:
NUnit.Framework.Assert.Zero(JobStat.Where(jS => (jS == BoSSS.Application.BoSSSpad.JobStatus.FailedOrCanceled)).Count(), "Some Jobs Failed");